# Day 06 Exercises — Solutions

Answer key for `day06_exercises.ipynb`. Some questions have more than one valid way to write them — these are the reference solutions, not the only correct answers.

**Before you start:** run `day06_python_sqlite.ipynb` first — this notebook connects to the `rawaj.sqlite` file it generates.

In [1]:
%load_ext sql

import sqlite3
import pandas as pd

sqlite_conn = sqlite3.connect("rawaj.sqlite")
%sql sqlite:///rawaj.sqlite --alias sqlite_rawaj

Connecting to 'sqlite_rawaj'

### Day 01

**1 — SELECT + LIMIT/OFFSET.** Show the 3rd-through-7th account managers, ordered alphabetically by name.

In [2]:
%%sql
SELECT manager_name FROM account_managers ORDER BY manager_name LIMIT 5 OFFSET 2;

Running query in 'sqlite_rawaj'

manager_name
Dina Rashed
Doaa El-Kholy
Fatma Rashed
Karim El-Nabawy
Magdy El-Masry


**2 — DISTINCT.** How many distinct governorates does the company operate in?

In [3]:
%%sql
SELECT DISTINCT governorate_name FROM governorates;

Running query in 'sqlite_rawaj'

governorate_name
Cairo
Giza
Alexandria
Qalyubia
Sharqia
Assiut


**3 — ORDER BY + COUNT.** How many total web events have ever been logged?

In [4]:
%%sql
SELECT COUNT(*) AS total_events FROM web_events;

Running query in 'sqlite_rawaj'

total_events
9000


### Day 02

**4 — WHERE + comparison operators.** Which orders sold for less than 500 EGP?

In [5]:
%%sql
SELECT order_id, customer_id, total_amount FROM orders WHERE total_amount < 500;

Running query in 'sqlite_rawaj'

order_id,customer_id,total_amount
27,493,401.62
34,906,224.08
50,1065,263.45
66,763,386.71
87,801,209.33
91,1015,340.04
100,628,238.32
112,1013,289.37
113,386,492.3
144,1177,200.36


**5 — AND / OR.** Which orders had a discount_amount over 500 EGP AND a total_amount over 10,000 EGP?

In [6]:
%%sql
SELECT order_id, discount_amount, total_amount FROM orders WHERE discount_amount > 500 AND total_amount > 10000;

Running query in 'sqlite_rawaj'

order_id,discount_amount,total_amount
148,624.28,11373.95
450,715.23,10190.07
666,1609.83,10499.26
747,561.87,15520.9
1239,556.82,11971.53
1499,2077.94,12642.81
1915,891.72,10300.53
2369,731.22,10666.26
2375,712.99,10050.86
2612,1564.3,13509.32


**6 — BETWEEN.** Which orders were placed during 2023?

In [7]:
%%sql
SELECT order_id, order_date FROM orders WHERE order_date BETWEEN '2023-01-01' AND '2023-12-31';

Running query in 'sqlite_rawaj'

order_id,order_date
3,2023-06-10 17:56:51
5,2023-08-12 12:42:08
6,2023-08-14 13:15:40
28,2023-11-23 20:13:34
29,2023-10-06 11:15:54
45,2023-11-04 13:15:50
48,2023-07-14 10:24:08
51,2023-10-08 09:13:04
63,2023-12-17 21:36:12
75,2023-06-09 12:34:27


**7 — IN.** Which governorates are Giza or Qalyubia?

In [8]:
%%sql
SELECT * FROM governorates WHERE governorate_name IN ('Giza', 'Qalyubia');

Running query in 'sqlite_rawaj'

governorate_id,governorate_name,manager_id
2,Giza,3.0
4,Qalyubia,13.0


**8 — NULL checks.** Are there any governorates with a missing (NULL) manager_id?

In [9]:
%%sql
SELECT COUNT(*) AS null_manager_governorates FROM governorates WHERE manager_id IS NULL;

Running query in 'sqlite_rawaj'

null_manager_governorates
1


**9 — LIKE.** Which sellers have a name containing 'el' (case-insensitive)?

In [10]:
%%sql
SELECT seller_name FROM sellers WHERE seller_name LIKE '%el%' LIMIT 5;

Running query in 'sqlite_rawaj'

seller_name
Heliopolis Mart
Heliopolis Bazaar
Adel El-Bahnasy Trading
Hassan El-Gendy Trading
Aya El-Gendy Store


**10 — GROUP BY + aggregates.** What's the average order value (total_amount) per customer, smallest first?

In [11]:
%%sql
SELECT customer_id, AVG(total_amount) AS avg_order FROM orders GROUP BY customer_id ORDER BY avg_order LIMIT 5;

Running query in 'sqlite_rawaj'

customer_id,avg_order
440,436.48
1041,437.97
203,471.21
222,679.33
1025,685.23


**11 — DATE functions.** How many orders were placed each month of 2023?

In [12]:
%%sql
SELECT STRFTIME('%m', order_date) AS mo, COUNT(*) AS num_orders FROM orders WHERE STRFTIME('%Y', order_date) = '2023' GROUP BY mo ORDER BY mo;

Running query in 'sqlite_rawaj'

mo,num_orders
06,176
07,189
08,165
09,197
10,176
11,165
12,182


### Day 03

**12 — INNER JOIN.** Combine every web event with the customer it belongs to.

In [13]:
%%sql
SELECT w.event_id, c.first_name, c.last_name FROM web_events w JOIN customers c ON w.customer_id = c.customer_id LIMIT 5;

Running query in 'sqlite_rawaj'

event_id,first_name,last_name
1,Ayman,El-Masry
2,Nesma,Hassanein
3,Salma,Hassanein
4,Karim,Hassanein
5,Manal,El-Bahnasy


**13 — Multi-table JOIN.** Trace a web event back to the governorate of the customer it belongs to.

In [14]:
%%sql
SELECT w.event_id, c.first_name || ' ' || c.last_name AS customer, g.governorate_name
FROM web_events w
JOIN customers c ON w.customer_id = c.customer_id
JOIN governorates g ON c.governorate_id = g.governorate_id
LIMIT 5;

Running query in 'sqlite_rawaj'

event_id,customer,governorate_name
1,Ayman El-Masry,Cairo
2,Nesma Hassanein,Alexandria
3,Salma Hassanein,Giza
4,Karim Hassanein,Cairo
5,Manal El-Bahnasy,Cairo


**14 — JOIN + GROUP BY.** Total web events per channel.

In [15]:
%%sql
SELECT channel, COUNT(*) AS num_events FROM web_events GROUP BY channel ORDER BY num_events DESC;

Running query in 'sqlite_rawaj'

channel,num_events
organic,2315
facebook,2252
google,1814
instagram,1787
direct,832


**15 — JOIN + HAVING.** Which customers have placed more than 5 orders?

In [16]:
%%sql
SELECT c.customer_id, c.first_name, c.last_name, COUNT(*) AS num_orders
FROM customers c JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name
HAVING COUNT(*) > 5
LIMIT 5;

Running query in 'sqlite_rawaj'

customer_id,first_name,last_name,num_orders
2,Nour,Fahmy,6
6,Salma,El-Masry,8
9,Omar,Zaki,6
13,Sara,El-Bahnasy,6
15,Nour,Hegazy,9


**16 — LEFT JOIN + anti-join.** Which customers have NEVER left a product review?

In [17]:
%%sql
SELECT c.customer_id, c.first_name, c.last_name
FROM customers c LEFT JOIN reviews r ON c.customer_id = r.customer_id
WHERE r.review_id IS NULL
LIMIT 5;

Running query in 'sqlite_rawaj'

customer_id,first_name,last_name
4,Mohamed,Zaki
5,Amr,Kandil
12,Nabil,Rashed
16,Heba,Mahmoud
29,Mona,Rashed


**17 — FULL JOIN via UNION.** Every customer/web-event pairing, matched or not.

In [18]:
%%sql
SELECT customers.customer_id AS customer_id, web_events.event_id AS event_id
FROM customers LEFT JOIN web_events ON customers.customer_id = web_events.customer_id
UNION
SELECT customers.customer_id, web_events.event_id
FROM customers RIGHT JOIN web_events ON customers.customer_id = web_events.customer_id
LIMIT 5;

Running query in 'sqlite_rawaj'

customer_id,event_id
1,7149
2,727
2,3619
2,5005
2,5051


**18 — CASE.** Tag every order 'Bulk' (subtotal 20,000+ EGP) or 'Standard'.

In [19]:
%%sql
SELECT order_id, subtotal, CASE WHEN subtotal >= 20000 THEN 'Bulk' ELSE 'Standard' END AS order_type FROM orders LIMIT 5;

Running query in 'sqlite_rawaj'

order_id,subtotal,order_type
1,1052.51,Standard
2,2051.01,Standard
3,1035.14,Standard
4,2353.84,Standard
5,4162.7,Standard


**19 — CASE + GROUP BY.** Bucket every account manager into a 'top' (1500+ orders) or 'not' performer flag.

In [20]:
%%sql
SELECT am.manager_name, COUNT(*) AS num_orders,
       CASE WHEN COUNT(*) > 1500 THEN 'top' ELSE 'not' END AS flag
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN governorates g ON c.governorate_id = g.governorate_id
JOIN account_managers am ON g.manager_id = am.manager_id
GROUP BY am.manager_name ORDER BY num_orders DESC LIMIT 5;

Running query in 'sqlite_rawaj'

manager_name,num_orders,flag
Samar Salah,2185,top
Ayman Zaki,1377,not
Karim El-Nabawy,891,not
Dina Rashed,625,not
Yara Soliman,592,not


**20 — CASE + HAVING.** Which customers fall specifically in the '5,000-10,000 EGP total spend' bracket?

In [21]:
%%sql
SELECT c.first_name || ' ' || c.last_name AS customer, SUM(o.total_amount) AS total_spend
FROM customers c JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name
HAVING SUM(o.total_amount) BETWEEN 5000 AND 10000
LIMIT 5;

Running query in 'sqlite_rawaj'

customer,total_spend
Amr Kandil,8947.29
Asmaa El-Nabawy,6850.73
Radwa Salah,6586.62
Hassan El-Masry,8308.69
Adel Fahmy,9317.05


**21 — String functions.** Clean a padded seller name, and build a slug from a governorate name.

In [22]:
%%sql
SELECT TRIM('   Al-Sayed Trading   ') AS trimmed,
       LOWER(SUBSTR('Alexandria', 1, 4)) || '-egypt' AS slug;

Running query in 'sqlite_rawaj'

trimmed,slug
Al-Sayed Trading,alex-egypt


**22 — COALESCE / IFNULL.** Fill in a friendly label wherever a computed bucket is NULL.

In [23]:
%%sql
SELECT COALESCE(NULL, 'no bucket') AS label;

Running query in 'sqlite_rawaj'

label
no bucket


### Day 04

**23 — Scalar subquery.** Which orders sold for less than the company-wide average?

In [24]:
%%sql
SELECT order_id, total_amount FROM orders WHERE total_amount < (SELECT AVG(total_amount) FROM orders) LIMIT 5;

Running query in 'sqlite_rawaj'

order_id,total_amount
1,1077.44
2,2104.36
3,1067.39
4,2410.83
6,1848.12


**24 — Row subquery.** Each customer's very FIRST web event ever (earliest occurred_at).

In [25]:
%%sql
SELECT customer_id, occurred_at, channel FROM web_events
WHERE (customer_id, occurred_at) IN (SELECT customer_id, MIN(occurred_at) FROM web_events GROUP BY customer_id)
ORDER BY customer_id LIMIT 5;

Running query in 'sqlite_rawaj'

customer_id,occurred_at,channel
1,2025-04-15 09:21:55,instagram
2,2023-06-20 14:20:54,instagram
3,2023-06-11 13:29:15,instagram
4,2023-09-05 18:13:04,organic
5,2024-02-07 11:52:23,facebook


**25 — Derived table.** Average order value, averaged across customers (average of each customer's own average).

In [26]:
%%sql
SELECT AVG(avg_order) AS avg_of_averages FROM (SELECT customer_id, AVG(total_amount) AS avg_order FROM orders GROUP BY customer_id) AS t;

Running query in 'sqlite_rawaj'

avg_of_averages
3847.559652999732


**26 — CTE.** Same average-of-averages question, written as a CTE.

In [27]:
%%sql
WITH t AS (SELECT customer_id, AVG(total_amount) AS avg_order FROM orders GROUP BY customer_id)
SELECT AVG(avg_order) AS avg_of_averages FROM t;

Running query in 'sqlite_rawaj'

avg_of_averages
3847.559652999732


**27 — Chained CTEs.** Which account managers sell above the average total sales per manager?

In [28]:
%%sql
WITH manager_totals AS (
    SELECT am.manager_id, am.manager_name, SUM(o.total_amount) AS total_sales
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN governorates g ON c.governorate_id = g.governorate_id
    JOIN account_managers am ON g.manager_id = am.manager_id
    GROUP BY am.manager_id, am.manager_name
),
avg_manager AS (
    SELECT AVG(total_sales) AS avg_sales FROM manager_totals
)
SELECT * FROM manager_totals WHERE total_sales > (SELECT avg_sales FROM avg_manager) LIMIT 5;

Running query in 'sqlite_rawaj'

manager_id,manager_name,total_sales
3,Ayman Zaki,5172425.08
5,Samar Salah,8469795.93


**28 — Temporary table.** Materialize the account manager with the FEWEST total sales, then query it twice.

In [29]:
%%sql
CREATE TEMPORARY TABLE bottom_manager_demo AS
SELECT am.manager_id, am.manager_name, SUM(o.total_amount) AS total_sales
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN governorates g ON c.governorate_id = g.governorate_id
JOIN account_managers am ON g.manager_id = am.manager_id
GROUP BY am.manager_id, am.manager_name
ORDER BY total_sales ASC LIMIT 1;

Running query in 'sqlite_rawaj'

++
||
++
++

**29 — View.** A 'bottom 10 customers by spend' view, for a churn-risk report.

In [30]:
%%sql
DROP VIEW IF EXISTS bottom10_customers_demo;

CREATE VIEW bottom10_customers_demo AS
SELECT c.customer_id, c.first_name, c.last_name, SUM(o.total_amount) AS total_sales
FROM orders o JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name
ORDER BY total_sales ASC LIMIT 10;

Running query in 'sqlite_rawaj'

++
||
++
++

**30 — Window function: running total.** Running total of discount_amount, ordered chronologically.

In [31]:
%%sql
SELECT order_date, discount_amount, SUM(discount_amount) OVER (ORDER BY order_date) AS running_total
FROM orders ORDER BY order_date LIMIT 5;

Running query in 'sqlite_rawaj'

order_date,discount_amount,running_total
2023-06-01 10:42:08,0.0,0.0
2023-06-01 15:33:21,0.0,0.0
2023-06-01 17:11:31,0.0,0.0
2023-06-01 20:07:14,43.35,43.35
2023-06-01 20:41:11,0.0,43.35


**31 — PARTITION BY.** Running total of subtotal, reset per customer.

In [32]:
%%sql
SELECT customer_id, order_date, SUM(subtotal) OVER (PARTITION BY customer_id ORDER BY order_date) AS customer_running_total
FROM orders ORDER BY customer_id, order_date LIMIT 5;

Running query in 'sqlite_rawaj'

customer_id,order_date,customer_running_total
1,2023-06-23 17:07:32,8459.3
1,2024-07-02 22:08:38,10641.4
1,2024-10-18 21:01:28,13229.31
1,2025-03-06 13:01:40,14859.769999999999
1,2025-03-23 23:02:43,22535.76


**32 — RANK / DENSE_RANK / ROW_NUMBER.** Rank account managers by number of orders handled, comparing how ties are handled.

In [33]:
%%sql
WITH totals AS (
    SELECT am.manager_name, COUNT(*) AS num_orders
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN governorates g ON c.governorate_id = g.governorate_id
    JOIN account_managers am ON g.manager_id = am.manager_id
    GROUP BY am.manager_name
)
SELECT manager_name, num_orders,
       RANK() OVER (ORDER BY num_orders DESC) AS rnk,
       DENSE_RANK() OVER (ORDER BY num_orders DESC) AS dense_rnk,
       ROW_NUMBER() OVER (ORDER BY num_orders DESC) AS row_num
FROM totals ORDER BY num_orders DESC LIMIT 5;

Running query in 'sqlite_rawaj'

manager_name,num_orders,rnk,dense_rnk,row_num
Samar Salah,2185,1,1,1
Ayman Zaki,1377,2,2,2
Karim El-Nabawy,891,3,3,3
Dina Rashed,625,4,4,4
Yara Soliman,592,5,5,5


**33 — Ranking + filtering.** Each account manager's single largest customer, by that customer's total spend.

In [34]:
%%sql
WITH customer_totals AS (
    SELECT am.manager_name, c.customer_id, c.first_name, c.last_name, SUM(o.total_amount) AS total_spend
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN governorates g ON c.governorate_id = g.governorate_id
    JOIN account_managers am ON g.manager_id = am.manager_id
    GROUP BY am.manager_name, c.customer_id, c.first_name, c.last_name
),
ranked AS (
    SELECT manager_name, first_name, last_name, total_spend,
           ROW_NUMBER() OVER (PARTITION BY manager_name ORDER BY total_spend DESC) AS r
    FROM customer_totals
)
SELECT manager_name, first_name, last_name, total_spend FROM ranked WHERE r = 1 ORDER BY manager_name LIMIT 5;

Running query in 'sqlite_rawaj'

manager_name,first_name,last_name,total_spend
Ayman Zaki,Radwa,Hassanein,57169.74
Dina Rashed,Amr,Younis,61063.45
Karim El-Nabawy,Hossam,Mahmoud,60579.869999999995
Samar Salah,Ghada,Aziz,56593.5
Yara Soliman,Eman,Younis,48021.6


**34 — LAG / LEAD.** Is the number of web events per month growing or shrinking?

In [35]:
%%sql
WITH monthly AS (SELECT STRFTIME('%Y-%m', occurred_at) AS month, COUNT(*) AS num_events FROM web_events GROUP BY month)
SELECT month, num_events, LAG(num_events) OVER (ORDER BY month) AS prev_month FROM monthly ORDER BY month LIMIT 5;

Running query in 'sqlite_rawaj'

month,num_events,prev_month
2023-06,259,None
2023-07,267,259
2023-08,294,267
2023-09,243,294
2023-10,288,243


**35 — Stored procedures.** Call the governorate-report procedure Day 05 built, for governorate 1. (SQLite has no stored procedures — write it as a plain Python function instead, same pattern as `day06_python_sqlite.ipynb`.)

In [36]:
def governorate_sales_report(governorate_id):
    query = (
        "SELECT g.governorate_name, COUNT(o.order_id) AS num_orders, "
        "SUM(o.total_amount) AS total_revenue "
        "FROM governorates g JOIN customers c ON c.governorate_id = g.governorate_id "
        "JOIN orders o ON o.customer_id = c.customer_id "
        "WHERE g.governorate_id = :governorate_id GROUP BY g.governorate_name"
    )
    return pd.read_sql(query, sqlite_conn, params={"governorate_id": governorate_id})

governorate_sales_report(1)

,governorate_name,num_orders,total_revenue
0,Cairo,2185,8469795.93


### Safe parameter binding

**36 — Bound parameters.** Write a function `customers_in_governorate(governorate_name)` that returns every customer in a given governorate, using a bound parameter (`:governorate_name`) rather than an f-string. Confirm it still returns the correct, safe result even if called with `"Cairo' OR '1'='1"`.

In [37]:
def customers_in_governorate(governorate_name):
    query = (
        "SELECT c.first_name, c.last_name FROM customers c "
        "JOIN governorates g ON c.governorate_id = g.governorate_id "
        "WHERE g.governorate_name = :governorate_name"
    )
    return pd.read_sql(query, sqlite_conn, params={"governorate_name": governorate_name})

customers_in_governorate("Cairo").head()

,first_name,last_name
0,Adel,Fawzy
1,Adel,Mahmoud
2,Adel,Salah
3,Adel,Soliman
4,Ahmed,Aziz


In [38]:
# a malicious-looking input — still returns the same safe result: zero matching rows,
# not every customer in the table
customers_in_governorate("Cairo' OR '1'='1")

,first_name,last_name
